<a href="https://colab.research.google.com/github/LeninGF/IAG-2024B-GenerativeQA/blob/fix-metric2/question-answering-Bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question Answering Generative

- Coder: Lenin G. Falconí
- Date: 2025-01-27

https://huggingface.co/docs/transformers/en/tasks/question_answering

## Instalación de Librerías

In [1]:
# For Colab
!pip install datasets
!pip install python-dotenv
!pip install huggingface_hub
!pip install datasets
!pip install evaluate # for colab

## Login en Huggingfaces

Se realiza el login usando archivo .env

In [2]:
# For Colab
from dotenv import load_dotenv
import os
dotenv_path = '/content/.env'
load_dotenv(dotenv_path)

True

In [3]:
import os
from huggingface_hub import login
token = os.getenv('HUGGINGFACE_TOKEN')
login(token)

In [4]:
# from huggingface_hub import notebook_login
# notebook_login()

## Carga de Librerías

In [5]:
import torch
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset
from pprint import pprint

## Carga del Dataset

Se procede a realizar la carga del dataset desde huggingface

In [6]:
path2dataset = "LeninGF/robos-question-answering"
dataset = load_dataset(path2dataset)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'context', 'question', 'answer_start', 'answer_end', 'impossible_find_answer', 'answer_text', 'is_impossible', 'context_id', 'answer_text_number_words'],
        num_rows: 4572
    })
})

In [8]:
# Dividir en train y test si es necesario
if "test" not in dataset:
    dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
    train_dataset = dataset["train"]
    test_dataset = dataset["test"]
else:
    train_dataset = dataset["train"]
    test_dataset = dataset["test"]

In [9]:
dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'context', 'question', 'answer_start', 'answer_end', 'impossible_find_answer', 'answer_text', 'is_impossible', 'context_id', 'answer_text_number_words'],
        num_rows: 3657
    })
    test: Dataset({
        features: ['index', 'context', 'question', 'answer_start', 'answer_end', 'impossible_find_answer', 'answer_text', 'is_impossible', 'context_id', 'answer_text_number_words'],
        num_rows: 915
    })
})

In [10]:
pprint(dataset["train"][0])

{'answer_end': 45,
 'answer_start': 25,
 'answer_text': '20 de junio del 2018',
 'answer_text_number_words': 5,
 'context': 'señor fiscal el dia ayer 20 de junio del 2018 a las 17h00 '
            'aproximadamente por el redondel de jaramijo donde esta la fabrica '
            'puerto mar del canton jaramijo yo iba caminando y hablando por '
            'telefono de repetente un sujeto que iva caminando me arrancho el '
            'telefono de maneta violenta y lego salio corriendo con mi '
            'telefono celular era una persona joven alta moreno y de '
            'contextura delgada solocito se pida al ecu 911 si existen camara '
            'en el lugar a fin de identificar a la persona que me robo anexo a '
            'mi denuncia factura del telefono donde consta sus caracteristicas',
 'context_id': 'context_93',
 'impossible_find_answer': False,
 'index': 466,
 'is_impossible': '0',
 'question': '¿En qué fecha ocurrió el incidente?'}


## Cargando Modelo Pre-Entrenado

Se considera utilizar los modelos:

- `dccuchile/bert-base-spanish-wwm-cased`
-  `mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es`

que sería un ajuste al Español

In [11]:
# 2. Cargar modelo y tokenizer en español
model_checkpoint = "mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es"  # Modelo BERT en español
# model_checkpoint = "dccuchile/bert-base-spanish-wwm-cased"  # Modelo BERT en español
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

Some weights of the model checkpoint at mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## Preprocesamiento

In [12]:
# 3. Corrección crítica: Preprocesamiento con batched=True
def prepare_train_features(examples):
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        max_length=384,
        truncation="only_second",
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    start_positions = []
    end_positions = []

    for i, sample_idx in enumerate(sample_map):
        if examples["impossible_find_answer"][sample_idx]:
            start_positions.append(0)
            end_positions.append(0)
            continue

        start_char = int(examples["answer_start"][sample_idx])
        end_char = int(examples["answer_end"][sample_idx])
        sequence_ids = inputs.sequence_ids(i)
        offsets = inputs["offset_mapping"][i]

        # Buscar el contexto (posición 1 en sequence_ids)
        context_start = sequence_ids.index(1)
        context_end = len(sequence_ids) - 1 - list(reversed(sequence_ids)).index(1)

        # Verificar si la respuesta está en el contexto
        if offsets[context_start][0] > end_char or offsets[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
            continue

        # Buscar posición de inicio
        start_idx = context_start
        while start_idx <= context_end and offsets[start_idx][0] <= start_char:
            start_idx += 1
        start_positions.append(start_idx - 1)

        # Buscar posición final
        end_idx = context_end
        while end_idx >= context_start and offsets[end_idx][1] >= end_char:
            end_idx -= 1
        end_positions.append(end_idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [13]:
# Aplicar preprocesamiento CORRECTO con batched=True
tokenized_train = train_dataset.map(
    prepare_train_features,
    batched=True,  # ¡Este es el cambio clave!
    remove_columns=train_dataset.column_names,
)

Revisando que la tokenizacion apunte a las respuestas

In [14]:
pprint(tokenized_train[0].keys())

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'start_positions', 'end_positions'])


In [15]:
len(tokenized_train[0]['input_ids'])

384

In [16]:
decoded_text = tokenizer.decode(tokenized_train[0]['input_ids'])
pprint(decoded_text)

('[CLS] ¿ en que fecha ocurrio el incidente? [SEP] senor fiscal el dia ayer 20 '
 'de junio del 2018 a las [UNK] aproximadamente por el redondel de jaramijo '
 'donde esta la fabrica puerto mar del canton jaramijo yo iba caminando y '
 'hablando por telefono de repetente un sujeto que iva caminando me arrancho '
 'el telefono de maneta violenta y lego salio corriendo con mi telefono '
 'celular era una persona joven alta moreno y de contextura delgada solocito '
 'se pida al ecu 911 si existen camara en el lugar a fin de identificar a la '
 'persona que me robo anexo a mi denuncia factura del telefono donde consta '
 'sus caracteristicas [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] '
 '[PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] '
 '[PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] '
 '[PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] '
 '[PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [P

In [17]:
start = tokenized_train[0]['start_positions']
end = tokenized_train[0]['end_positions']
print(start, end)
tokenizer.decode(tokenized_train[0]['input_ids'][start:end+1])

17 21


'20 de junio del 2018'

In [18]:
tokenized_test = test_dataset.map(
    prepare_train_features,
    batched=True,  # Procesar una muestra a la vez
    remove_columns=test_dataset.column_names,  # Eliminar columnas originales
)

Map:   0%|          | 0/915 [00:00<?, ? examples/s]

In [19]:
# from transformers import DefaultDataCollator

# data_collator = DefaultDataCollator()

In [20]:
# from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

# model = AutoModelForQuestionAnswering.from_pretrained("distilbert/distilbert-base-uncased")

## Evaluación del Rendimiento del Modelo
Se utilizara `squad_metric` para calcular EM y F1

In [21]:
# 4. Función de métricas optimizada
squad_metric = evaluate.load("squad_v2")

def compute_metrics(p):
    predictions = p.predictions
    if isinstance(predictions, tuple):
        start_logits, end_logits = predictions
    else:
        start_logits = predictions["start_logits"]
        end_logits = predictions["end_logits"]

    # Convertir logits a posiciones
    start_pred = np.argmax(start_logits, axis=1)
    end_pred = np.argmax(end_logits, axis=1)

    # Preparar respuestas
    formatted_predictions = []
    for i, (start, end) in enumerate(zip(start_pred, end_pred)):
        example = test_dataset[i]
        context = example["context"]
        offsets = tokenized_test[i]["offset_mapping"]

        if start >= len(offsets) or end >= len(offsets):
            answer = ""
        else:
            answer = context[offsets[start][0]:offsets[end][1]]

        formatted_predictions.append({
            "id": str(example["index"]),
            "prediction_text": answer,
            "no_answer_probability": 0.0 if answer else 1.0
        })

    # Preparar referencias
    references = [{
        "id": str(ex["index"]),
        "answers": {
            "text": [ex["answer_text"]],
            "answer_start": [ex["answer_start"]]
        },
        # "is_impossible": ex["impossible_find_answer"]
    } for ex in test_dataset]

    return squad_metric.compute(predictions=formatted_predictions, references=references)

Para evaluar durante el entrenamiento se defina la clase `QATrainer`

## Entrenamiento

In [22]:
os.environ["WANDB_DISABLED"] = "true"

In [27]:
training_args = TrainingArguments(
    output_dir="./fge-robos-qa-model", # colocar dentro de models
    evaluation_strategy="epoch",
    learning_rate=2e-5,  # Slightly lower for non-English models
    per_device_train_batch_size=64,  # Adjust based on GPU memory
    per_device_eval_batch_size=64,
    num_train_epochs=10,  # Spanish datasets may need more epochs
    weight_decay=0.01,
    save_strategy="epoch",
    fp16=True,  # Use if GPU supports it
    logging_dir="./logs",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [28]:
# 6. Crear el Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

<ipython-input-28-af6bac14e49d>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [29]:
# 7. Evaluación inicial (antes del fine-tuning)
print("Evaluación antes del fine-tuning:")
initial_eval = trainer.evaluate()
pprint(initial_eval)

Evaluación antes del fine-tuning:


{'eval_HasAns_exact': 53.55191256830601,
 'eval_HasAns_f1': 81.7040977184179,
 'eval_HasAns_total': 915,
 'eval_best_exact': 53.55191256830601,
 'eval_best_exact_thresh': 0.0,
 'eval_best_f1': 81.7040977184179,
 'eval_best_f1_thresh': 0.0,
 'eval_exact': 53.55191256830601,
 'eval_f1': 81.7040977184179,
 'eval_loss': 0.9101653099060059,
 'eval_model_preparation_time': 0.0077,
 'eval_runtime': 12.3982,
 'eval_samples_per_second': 73.801,
 'eval_steps_per_second': 1.21,
 'eval_total': 915}


In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time,Exact,F1,Total,Hasans Exact,Hasans F1,Hasans Total,Best Exact,Best Exact Thresh,Best F1,Best F1 Thresh
1,No log,0.896278,0.007700,54.207650,81.428103,915,54.207650,81.428103,915,54.207650,0.000000,81.428103,0.000000
2,No log,0.957770,0.007700,54.972678,82.369370,915,54.972678,82.369370,915,54.972678,0.000000,82.369370,0.000000
3,No log,1.008776,0.007700,55.737705,82.880522,915,55.737705,82.880522,915,55.737705,0.000000,82.880522,0.000000
4,No log,1.086515,0.007700,54.426230,81.745943,915,54.426230,81.745943,915,54.426230,0.000000,81.745943,0.000000
5,No log,1.203394,0.007700,53.770492,81.532838,915,53.770492,81.532838,915,53.770492,0.000000,81.532838,0.000000
6,No log,1.282165,0.007700,54.207650,81.998528,915,54.207650,81.998528,915,54.207650,0.000000,81.998528,0.000000
7,No log,1.335699,0.007700,54.207650,81.729399,915,54.207650,81.729399,915,54.207650,0.000000,81.729399,0.000000
8,No log,1.373773,0.007700,54.644809,81.952552,915,54.644809,81.952552,915,54.644809,0.000000,81.952552,0.000000
9,0.429200,1.421485,0.007700,54.754098,81.738464,915,54.754098,81.738464,915,54.754098,0.000000,81.738464,0.000000
10,0.429200,1.434188,0.007700,53.442623,81.372865,915,53.442623,81.372865,915,53.442623,0.000000,81.372865,0.000000


TrainOutput(global_step=580, training_loss=0.3987290382385254, metrics={'train_runtime': 840.8625, 'train_samples_per_second': 43.491, 'train_steps_per_second': 0.69, 'total_flos': 7166716795376640.0, 'train_loss': 0.3987290382385254, 'epoch': 10.0})

## Evaluación


In [31]:
# Evaluate on test set
print("\nEvaluación final:")
final_eval = trainer.evaluate()
pprint(final_eval)


Evaluación final:


{'epoch': 10.0,
 'eval_HasAns_exact': 55.73770491803279,
 'eval_HasAns_f1': 82.88052227892325,
 'eval_HasAns_total': 915,
 'eval_best_exact': 55.73770491803279,
 'eval_best_exact_thresh': 0.0,
 'eval_best_f1': 82.88052227892325,
 'eval_best_f1_thresh': 0.0,
 'eval_exact': 55.73770491803279,
 'eval_f1': 82.88052227892325,
 'eval_loss': 1.0087758302688599,
 'eval_model_preparation_time': 0.0077,
 'eval_runtime': 12.8109,
 'eval_samples_per_second': 71.424,
 'eval_steps_per_second': 1.171,
 'eval_total': 915}


## Guardando Modelo y Tokenizer

In [32]:
# os.mkdir("fge-qa-model")
# guardando localmente
model.save_pretrained(".fge-qa-model/fine-tuned-qa-model")
tokenizer.save_pretrained(".fge-qa-model/fine-tuned-qa-model")

('.fge-qa-model/fine-tuned-qa-model/tokenizer_config.json',
 '.fge-qa-model/fine-tuned-qa-model/special_tokens_map.json',
 '.fge-qa-model/fine-tuned-qa-model/vocab.txt',
 '.fge-qa-model/fine-tuned-qa-model/added_tokens.json',
 '.fge-qa-model/fine-tuned-qa-model/tokenizer.json')

In [33]:
trainer.push_to_hub()

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.30k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/LeninGF/fge-robos-qa-model/commit/45e264a1405407866f2d8748085f581512149516', commit_message='End of training', commit_description='', oid='45e264a1405407866f2d8748085f581512149516', pr_url=None, repo_url=RepoUrl('https://huggingface.co/LeninGF/fge-robos-qa-model', endpoint='https://huggingface.co', repo_type='model', repo_id='LeninGF/fge-robos-qa-model'), pr_revision=None, pr_num=None)

## Demo

In [36]:
import numpy as np
from pprint import pprint
preguntas_comunes = [
    "¿Qué objetos fueron robados?",
    "¿En qué fecha ocurrió el incidente?",
    "¿A qué hora sucedió el robo?",
    "¿En qué dirección o entre qué calles sucedió el robo, sucedo o incidente?",
    "¿Qué valor en dólares tenían los objetos sustraídos o robados?",
    # "¿Existió intimidación, agresión o violencia?"
]
# context = "es el caso señor fiscal que el dia de hoy 28 de julio del 2016 siendo aproximadamente las 17h00 en circunstancias que me baje de un bus en la parroquia san camilo con la finalidad de dirigirme a mi lugar de trabajo esto el taller eco frio de repente al llegar a la altura del cuerpo de bomberos fui inteceptado por dos sujetos inidentificados que se movilizaban a bordo de una motocicleta marca suzuki colo rojo sin placas los mismos que con un arma de fuego me intimidaron acto seguido procedieron a robarme ciento ochenta dolares en efectivo dinero que era de producto de mi trabajo luego se dieron a la fuga con rumbo desconocido por tal motivo solicito se realicen las respectivas investigaciones"
random_idx = np.random.randint(0, dataset["test"].num_rows)
context = dataset["test"][random_idx]["context"]
print(f"Test sample: {random_idx}")
pprint(context)

Test sample: 435
('el día de ayer 17 de junio del 2020 aproximadamente a las 10h00 estaba '
 'realizando una factura en el sector de cotocollao av legarda instantes en '
 'los cuales fui sorprendido por un sujeto de acento costeño mismos que bajo '
 'amenazas y con una cuchillo procedido a sustraerme lo siguiente un celular '
 'marca samsun modelo a10 s duo color negro de la operadora movistar asignado '
 'con el número 0995781523 imei 358099 10 7234734 perteneciente a la empresa '
 'alimentos yupi con lo antes expuesto solicito las respectivas '
 'investigaciones para mayor información comunicarse a los números 0983771221')


In [37]:
from transformers import pipeline

question_answerer = pipeline("question-answering", model="LeninGF/fge-robos-qa-model")
for question in preguntas_comunes:
    answer = question_answerer(question=question, context=context)
    pprint(question)
    pprint(answer)

Device set to use cuda:0


'¿Qué objetos fueron robados?'
{'answer': 'un celular marca samsun modelo a10 s duo color negro',
 'end': 336,
 'score': 0.9661497473716736,
 'start': 284}
'¿En qué fecha ocurrió el incidente?'
{'answer': 'el día de ayer 17 de junio del 2020',
 'end': 35,
 'score': 0.615260899066925,
 'start': 0}
'¿A qué hora sucedió el robo?'
{'answer': 'aproximadamente a las 10h00',
 'end': 63,
 'score': 0.9836047887802124,
 'start': 36}
'¿En qué dirección o entre qué calles sucedió el robo, sucedo o incidente?'
{'answer': 'sector de cotocollao av legarda',
 'end': 131,
 'score': 0.483816534280777,
 'start': 100}
'¿Qué valor en dólares tenían los objetos sustraídos o robados?'
{'answer': '10 7234734', 'end': 418, 'score': 0.16715489327907562, 'start': 408}


## Resultados a Latex

Se utiliza el siguiente código para generar archivos `.tex` a fin de poder guardar o presentar los resultados antes y después del fine tuning

In [38]:
import pandas as pd

df_antes = pd.DataFrame(list(initial_eval.items()), columns=['Metric', 'Value'])
df_despues = pd.DataFrame(list(final_eval.items()), columns=['Metric', 'Value'])
latex_antes = df_antes.to_latex(index=False, float_format="%.4f")
latex_despues = df_despues.to_latex(index=False, float_format="%.4f")

with open('resultados_antes.tex', 'w') as f:
    f.write(latex_antes)

with open('resultados_despues.tex', 'w') as f:
    f.write(latex_despues)


# TODO

- Evaluar el modelo sin hacer fine tuning
- Generar un branch para cambios del programa
- Incluir otras metricas de evaluacion para el entrenamiento
